# Delta-E Backend Comparison: CIEDE2000 vs Oklab

Every perceptual claim renoir makes rests on a delta-E formula: the function that answers “how different do these two colors look?”. Since v3.4 the package has used CIEDE2000 (Sharma, Wu & Dalal, 2005), the current CIE standard. Oklab (Ottosson, 2020) is a newer perceptually uniform color space in which plain Euclidean distance plays the same role with simpler math.

This lesson asks an empirical question: does the choice of backend change the answers? We compute PEMD and CCI under both backends on three art-historical palettes and compare the results.

In [1]:
from renoir.color import ColorAnalyzer

analyzer = ColorAnalyzer()

## Three art-historical palettes

Three five-color palettes with proportions, loosely inspired by three painters: a dark-ground palette of ultramarine, ochre, and pearl (Vermeer), a light high-key garden palette (Monet), and a primary-triad palette (Mondrian).

In [2]:
vermeer = [
    ((38, 64, 120), 0.30),
    ((201, 162, 75), 0.25),
    ((238, 232, 220), 0.20),
    ((28, 24, 20), 0.15),
    ((179, 58, 48), 0.10),
]

monet = [
    ((139, 169, 132), 0.30),
    ((164, 191, 208), 0.25),
    ((158, 150, 187), 0.20),
    ((217, 175, 185), 0.15),
    ((73, 102, 78), 0.10),
]

mondrian = [
    ((221, 28, 26), 0.30),
    ((35, 68, 140), 0.25),
    ((250, 201, 34), 0.20),
    ((240, 240, 235), 0.15),
    ((25, 25, 25), 0.10),
]

## PEMD under both backends

Palette Earth Mover’s Distance combines color similarity with proportion differences. Watch the scale: CIEDE2000 measures black-to-white as 100, Oklab as 1.0, so Oklab values sit roughly two orders of magnitude lower. The absolute numbers are not comparable across backends; the *ranking* of palette pairs is what matters.

In [3]:
pairs = [("Vermeer-Monet", vermeer, monet), ("Vermeer-Mondrian", vermeer, mondrian), ("Monet-Mondrian", monet, mondrian)]

for metric in ("cie2000", "oklab"):
    print(f"--- {metric} ---")
    for label, p1, p2 in pairs:
        d = analyzer.palette_earth_movers_distance(p1, p2, distance_metric=metric)
        print(f"  {label}: {d:.4f}")

--- cie2000 ---
  Vermeer-Monet: 30.6159
  Vermeer-Mondrian: 13.6699
  Monet-Mondrian: 34.1439
--- oklab ---
  Vermeer-Monet: 0.2361
  Vermeer-Mondrian: 0.1105
  Monet-Mondrian: 0.2565


Do both backends agree on which pair is closest and which is most distant? If the order matches, the backends agree on the structure of palette space for these examples; disagreements would mark cases where the blue-region non-uniformity of CIELAB (or Oklab’s own approximations) changes the conclusion.

## CCI under both backends

Only the perceptual-spread component of the Color Complexity Index depends on the delta-E backend. CCI normalizes spread by the metric’s black-to-white distance (100 for CIEDE2000, 1 for Oklab), so the composite score stays on a comparable 0-1 scale either way.

In [4]:
palettes = {"Vermeer": vermeer, "Monet": monet, "Mondrian": mondrian}

for name, palette in palettes.items():
    colors = [c for c, _ in palette]
    props = [w for _, w in palette]
    for metric in ("cie2000", "oklab"):
        r = analyzer.calculate_color_complexity(colors, proportions=props, distance_metric=metric)
        print(f"{name} [{metric}]: CCI={r['cci']:.3f}  spread={r['perceptual_spread']:.3f}  entropy={r['hue_entropy']:.3f}")

Vermeer [cie2000]: CCI=0.258  spread=0.477  entropy=0.382
Vermeer [oklab]: CCI=0.233  spread=0.395  entropy=0.382
Monet [cie2000]: CCI=0.267  spread=0.308  entropy=0.648
Monet [oklab]: CCI=0.228  spread=0.178  entropy=0.648
Mondrian [cie2000]: CCI=0.342  spread=0.538  entropy=0.536
Mondrian [oklab]: CCI=0.316  spread=0.449  entropy=0.536


## Optional: CAM16-UCS backend

With the colour-science extra installed (`pip install 'renoir-wikiart[cam16]'`), a third backend becomes available: CAM16-UCS (Li et al., 2017), which models chromatic adaptation and viewing conditions. The cell below runs either way; without the extra it prints a note.

In [5]:
try:
    import colour  # noqa: F401
except ImportError:
    print("CAM16-UCS backend not available; install with: pip install 'renoir-wikiart[cam16]'")
else:
    for label, p1, p2 in pairs:
        d = analyzer.palette_earth_movers_distance(p1, p2, distance_metric="cam16")
        print(f"  {label}: {d:.4f}")


  Vermeer-Monet: 31.1541
  Vermeer-Mondrian: 16.2288
  Monet-Mondrian: 36.6883


## Takeaways

- The backend choice changes absolute values but, on these palettes, leaves the broad structure (which palettes are close, which are complex) intact.
- CIEDE2000 remains the standards-backed default; Oklab is cheaper to compute and avoids the CIELAB blue-region non-uniformity.
- For corpus studies, run both and report where rankings disagree; those cases are the scientifically interesting ones.


## References

Li, C., Li, Z., Wang, Z., Xu, Y., Luo, M. R., Cui, G., Melgosa, M., Brill, M. H., & Pointer, M. (2017). Comprehensive color solutions: CAM16, CAT16, and CAM16-UCS. *Color Research & Application*, *42*(6), 703–718. https://doi.org/10.1002/col.22131

Ottosson, B. (2020, December 23). *A perceptual color space for image processing*. https://bottosson.github.io/posts/oklab/

Sharma, G., Wu, W., & Dalal, E. N. (2005). The CIEDE2000 color-difference formula: Implementation notes, supplementary test data, and mathematical observations. *Color Research & Application*, *30*(1), 21–30. https://doi.org/10.1002/col.20070